# logit-thermo A2-5：分析（CPU）

- **必要な secret：なし**。**ランタイム：CPU（標準）**。見込みは 20〜40 分。
- `A2_5/` の 9 回と、A2-2 の 7 モデル（記録用）を読み、ゲート v0.2 の判定を計算して `A2_5/a2_5_summary.json` に書きます。**結果は表示しません。**
- 計算のコード：`scripts/a2_2_collect.py`・`scripts/a2_2_analyze.py`・`scripts/a2_explore_copy.py` は Drive の前のノートブックから取り出してハッシュで確かめ、`scripts/a2_5_analyze.py`（`f4ef35b9f594a205…`）は埋め込み（データを開く前に固定）。
- **通知**：🚀・🎉・⚠️。名札は「logit-thermo A2-5 分析（判定と要約、CPU）」。終わったら、またはエラーで止まったら、ランタイムを切ります。


In [ ]:
# セル1：通知、GPU、Drive
import os, time, urllib.request
NTFY_TOPIC = 'YOUR_NTFY_TOPIC'
NB_LABEL = 'logit-thermo A2-5 分析（判定と要約、CPU）'
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=(msg + '\n[' + NB_LABEL + ']').encode('utf-8'),
                                     headers={'Title': 'logit-thermo - A2-5 analyze', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
_NOTIFIED = {'error': False}
def _on_error(result):
    if result.error_in_exec is not None and not _NOTIFIED['error']:
        _NOTIFIED['error'] = True
        import traceback
        e = result.error_in_exec
        txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
        v = os.environ.get('HF_TOKEN')
        if v: txt = txt.replace(v, '***')
        txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
        notify('⚠️ 止まりました（エラー）：\n' + txt, tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('🚀 スタートしました（これが届けば通知の設定は OK です）', tags='rocket')

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# セル2：分析のコード
import ast, glob, hashlib, json
def take_src(want):
    """want：{変数の名前: sha256}。Drive の logit-thermo のノートブックから、埋め込んだコードを取り出してハッシュで確かめる。"""
    got = {}
    for p in sorted(glob.glob('/content/drive/MyDrive/logit-thermo' + '/**/*.ipynb', recursive=True)):
        try:
            cells = json.load(open(p))['cells']
        except Exception:
            continue
        for c in cells:
            s = c['source'] if isinstance(c['source'], str) else ''.join(c['source'])
            if c.get('cell_type') != 'code' or 'SRC' not in s:
                continue
            try:
                body = ast.parse(s).body
            except SyntaxError:
                continue
            for n in body:
                if isinstance(n, ast.Assign) and isinstance(n.targets[0], ast.Name) and n.targets[0].id in want and n.targets[0].id not in got:
                    v = ast.literal_eval(n.value)
                    if hashlib.sha256(v.encode()).hexdigest() == want[n.targets[0].id]:
                        got[n.targets[0].id] = v
        if len(got) == len(want):
            break
    miss = [k for k in want if k not in got]
    assert not miss, ('Drive のノートブックからコードを取り出せませんでした（前のノートブックを動かしていませんか）：', miss)
    return got
_s = take_src({'SRC22': 'e578cfc480980be6b0d1e18377b6beb8b7555e0744f56b13f63cb304a9f73d53', 'SRCA22': 'fe1f96db1f7e84031e85a70bcc8dcbb95c9fc06b3367657ff4074a9f9d1343f7', 'SRCX': '012c6dc1fb2ac8a3d120fa2c35a14480c7dc82815e6775cb3829a8bf16284759'})
SRC22, SRCA22, SRCX = _s['SRC22'], _s['SRCA22'], _s['SRCX']
SRC5A = '"""A2-5（ゲート docs/A2_5_GATE.md v0.2）の分析。データを開く前に固定する。CPU。\nscripts/a2_2_collect.py・scripts/a2_2_analyze.py・scripts/a2_explore_copy.py を実行したのと同じ名前空間で実行する\n（split_doc・per_doc_feedback・feedback_stats・crossing_mono・distinct4・copy_flags・decompose を変えずに使う）。\n\n    analyze5(a2_5 のフォルダ, a2_2 のフォルダ, docs) -> dict\n"""\nimport json, math\nimport numpy as np\n\nA5_SIZE = [(\'pythia-70m\', 70e6), (\'pythia-160m\', 160e6), (\'pythia-410m\', 410e6), (\'pythia-1b\', 1.0e9),\n           (\'pythia-1.4b\', 1.4e9), (\'pythia-2.8b\', 2.8e9)]                       # 名目のパラメータの数（名前のとおり）\nA5_STEP = [(\'pythia-1b@step8000\', 8000), (\'pythia-1b@step24000\', 24000), (\'pythia-1b@step64000\', 64000), (\'pythia-1b\', 143000)]\nA5_A22 = [(\'pythia-1b\', \'EleutherAI/pythia-1b\'), (\'qwen2.5-0.5b\', \'Qwen/Qwen2.5-0.5B\'), (\'qwen2.5-0.5b-inst\', \'Qwen/Qwen2.5-0.5B-Instruct\'),\n          (\'qwen2.5-1.5b\', \'Qwen/Qwen2.5-1.5B\'), (\'olmo2-1b\', \'allenai/OLMo-2-0425-1B\'), (\'smollm2-1.7b\', \'HuggingFaceTB/SmolLM2-1.7B\'),\n          (\'smollm2-1.7b-inst\', \'HuggingFaceTB/SmolLM2-1.7B-Instruct\')]\nA5_BAND = (128, 512)\nA5_FMAX, A5_MIN_DOCS = -0.2, 30\nA5_BOOT, A5_BOOT_SEED = 1000, 20261103\nA5_TSTAR = 0.8\n\n\ndef rank(x):\n    x = np.asarray(x, float)\n    o = np.argsort(x, kind=\'mergesort\'); r = np.empty(len(x)); r[o] = np.arange(len(x), dtype=float)\n    for v in np.unique(x):\n        m = x == v; r[m] = r[m].mean()\n    return r\n\n\ndef spearman(a, b):\n    a, b = np.asarray(a, float), np.asarray(b, float)\n    if len(a) < 2 or np.all(a == a[0]) or np.all(b == b[0]):\n        return float(\'nan\')\n    return float(np.corrcoef(rank(a), rank(b))[0, 1])\n\n\ndef doc_parts(Z, prompts, humans):\n    """→ glass, crystal, f：(nD, nT)。decompose が外した記事・温度は nan。帯は A5_BAND。"""\n    toks, gen, tf = Z[\'toks\'], Z[\'gen\'][0], Z[\'tf\'][0]\n    nT, S, nD, L = toks.shape\n    a, b = A5_BAND\n    hum = [copy_flags(prompts[j], humans[j]) for j in range(nD)]\n    out = np.full((3, nD, nT), np.nan)\n    for k in range(nT):\n        for j in range(nD):\n            gfl = np.stack([copy_flags(prompts[j], toks[k, s, j]) for s in range(S)])\n            r = decompose(gen[k, :, j, a:b], gfl[:, a:b], tf[j, k, a:b], hum[j][a:b])\n            if r is not None:\n                out[:, j, k] = r[\'glass\'], r[\'crystal\'], r[\'f\']\n    return out\n\n\ndef G_at(parts, k, idx=None):\n    """温度の番号 k での G。読めなければ nan。→ (G, ガラスの平均, f の平均, 使えた記事の数)"""\n    g, f = parts[0][:, k], parts[2][:, k]\n    if idx is not None:\n        g, f = g[idx], f[idx]\n    ok = np.isfinite(f)\n    n = int(ok.sum())\n    if n == 0:\n        return float(\'nan\'), float(\'nan\'), float(\'nan\'), 0\n    mg, mf = float(g[ok].mean()), float(f[ok].mean())\n    G = mg / mf if (n >= A5_MIN_DOCS and mf <= A5_FMAX) else float(\'nan\')\n    return G, mg, mf, n\n\n\ndef G_rel_at(parts, temps, T0, idx=None):\n    if not np.isfinite(T0):\n        return float(\'nan\')\n    Ts = A5_TSTAR * T0\n    lt = np.log(temps)\n    if not (temps[0] <= Ts <= temps[-1]):\n        return float(\'nan\')\n    i = int(np.searchsorted(lt, math.log(Ts), side=\'right\') - 1); i = min(i, len(temps) - 2)\n    w = (math.log(Ts) - lt[i]) / (lt[i + 1] - lt[i])\n    _, g0, f0, n0 = G_at(parts, i, idx); _, g1, f1, n1 = G_at(parts, i + 1, idx)\n    if min(n0, n1) < A5_MIN_DOCS:\n        return float(\'nan\')\n    mg, mf = (1 - w) * g0 + w * g1, (1 - w) * f0 + w * f1\n    return mg / mf if mf <= A5_FMAX else float(\'nan\')\n\n\ndef axis_word(xs, ys_point, ys_boot, need_all, strict_one, pos, neg):\n    """xs：軸の値、ys_point：点の値（nan は読めない）、ys_boot：(B, 点の数)。→ (記号, 詳細)"""\n    xs = np.asarray(xs, float); yp = np.asarray(ys_point, float)\n    ok = np.isfinite(yp)\n    if (need_all and not ok.all()) or ok.sum() < (len(xs) if need_all else 4):\n        return \'なし\', dict(rho=float(\'nan\'), n=int(ok.sum()), frac_pos=float(\'nan\'), frac_neg=float(\'nan\'))\n    rho = spearman(xs[ok], yp[ok])\n    sp, sn = 0, 0\n    for yb in ys_boot:\n        okb = np.isfinite(yb)\n        if (need_all and not okb.all()) or okb.sum() < (len(xs) if need_all else 4):\n            continue\n        r = spearman(xs[okb], yb[okb])\n        sp += int(r > 0); sn += int(r < 0)\n    B = len(ys_boot)\n    fp, fn = sp / B, sn / B\n    th = (1.0 if strict_one else 0.9) - 1e-9                  # 浮動小数の誤差で線引きの上の値が外れないように\n    sym = \'＋\' if (rho >= th and fp >= 0.95) else \'−\' if (rho <= -th and fn >= 0.95) else \'なし\'\n    return sym, dict(rho=rho, n=int(ok.sum()), frac_pos=fp, frac_neg=fn)\n\n\ndef words(sym_a, sym_b, pos, neg):\n    if sym_a == sym_b == \'＋\':\n        return pos\n    if sym_a == sym_b == \'−\':\n        return neg\n    return \'はっきりしない\'\n\n\ndef model_block(Z, prompts, humans, temps):\n    parts = doc_parts(Z, prompts, humans)\n    fd = per_doc_feedback(Z[\'tf\'][0], Z[\'gen\'][0])\n    fs = feedback_stats(fd, temps, np.random.default_rng(0))\n    return parts, fd, fs\n\n\ndef analyze5(d5, d2, docs):\n    from transformers import AutoTokenizer\n    tokp = AutoTokenizer.from_pretrained(\'EleutherAI/pythia-1b\')\n    pairs = [split_doc(tokp, d[\'text\']) for d in docs]\n    names = sorted({n for n, _ in A5_SIZE} | {n for n, _ in A5_STEP})\n    res = {\'models\': {}, \'missing\': []}\n    blocks = {}\n    for n in names:\n        try:\n            Z = (lambda z: {k: z[k] for k in z.files})(np.load(f\'{d5}/{n}.npz\'))\n        except FileNotFoundError:\n            res[\'missing\'].append(n); continue\n        use = [int(u) for u in Z[\'use\']]\n        temps = [float(t) for t in Z[\'temps\']]\n        prompts = [pairs[i][0] for i in use]; humans = [pairs[i][1] for i in use]\n        parts, fd, fs = model_block(Z, prompts, humans, temps)\n        blocks[n] = (parts, fd, fs, use)\n        k07, k08 = temps.index(0.7), temps.index(0.8)\n        G07, g07, f07, n07 = G_at(parts, k07)\n        G08, _, _, _ = G_at(parts, k08)\n        res[\'models\'][n] = dict(T0=fs[\'T0\'], T0_ci=fs[\'T0_ci\'], eps=fs[\'eps\'], G07=G07, G08=G08, G_rel=G_rel_at(parts, temps, fs[\'T0\']),\n                                glass07=g07, f07=f07, crystal07=float(np.nanmean(parts[1][:, k07])), n_docs07=n07,\n                                copy4_07=float(np.mean([copy_flags(prompts[j], Z[\'toks\'][k07, s, j])[A5_BAND[0]:].mean()\n                                                        for s in range(Z[\'toks\'].shape[1]) for j in range(len(use))])),\n                                distinct4=distinct4(Z[\'toks\']), ent_T1_human=float(np.mean(Z[\'tf\'][2][:, temps.index(1.0), :])))\n    uses = {tuple(b[3]) for b in blocks.values()}\n    assert len(uses) == 1, \'使った記事が回によって違う\'\n    nD = len(next(iter(uses)))\n    rng = np.random.default_rng(A5_BOOT_SEED)\n    boots = [rng.integers(0, nD, nD) for _ in range(A5_BOOT)]\n\n    def boot_vals(n):\n        parts, fd, fs, use = blocks[n]\n        out = np.full((A5_BOOT, 3), np.nan)                       # G07, G_rel, log T0\n        k07 = temps.index(0.7)\n        for b, idx in enumerate(boots):\n            T0b = crossing_mono(np.median(fd[idx], 0), temps, fs[\'eps\'])\n            out[b] = G_at(parts, k07, idx)[0], G_rel_at(parts, temps, T0b, idx), math.log(T0b) if np.isfinite(T0b) else np.nan\n        return out\n    BV = {n: boot_vals(n) for n in blocks}\n    # 確かめ：ブートストラップの元の値の統計\n    res[\'boot_ci\'] = {n: {k: [float(np.nanpercentile(BV[n][:, q], 2.5)), float(np.nanpercentile(BV[n][:, q], 97.5))]\n                          for q, k in enumerate([\'G07\', \'G_rel\', \'logT0\'])} for n in BV}\n\n    def axis(spec, need_all, strict_one):\n        ns = [n for n, _ in spec if n in blocks]\n        xs = [math.log(v) if not strict_one else v for n, v in spec if n in blocks]\n        out = {}\n        for q, key in enumerate([\'G07\', \'G_rel\', \'logT0\']):\n            yp = [res[\'models\'][n][key] if key != \'logT0\' else (math.log(res[\'models\'][n][\'T0\']) if np.isfinite(res[\'models\'][n][\'T0\']) else np.nan) for n in ns]\n            yb = np.stack([BV[n][:, q] for n in ns], 1)\n            out[key] = axis_word(xs, yp, yb, need_all, strict_one, None, None)\n        return ns, out\n    ns1, a1 = axis(A5_SIZE, need_all=False, strict_one=False)\n    ns2, a2 = axis(A5_STEP, need_all=True, strict_one=True)\n    if len(ns2) < len(A5_STEP):\n        a2 = {k: (\'なし\', dict(note=\'学びの軸の点が欠けている\')) for k in a2}\n    res[\'axes\'] = {\'size\': {\'models\': ns1, **{k: dict(sym=v[0], **v[1]) for k, v in a1.items()}},\n                   \'step\': {\'models\': ns2, **{k: dict(sym=v[0], **v[1]) for k, v in a2.items()}}}\n    res[\'Q1\'] = words(a1[\'G07\'][0], a1[\'G_rel\'][0], \'大きいほどガラス寄り\', \'大きいほど結晶寄り\')\n    res[\'Q2\'] = words(a2[\'G07\'][0], a2[\'G_rel\'][0], \'学ぶほどガラス寄り\', \'学ぶほど結晶寄り\')\n    s3 = a1[\'logT0\'][0]; s4 = a2[\'logT0\'][0]\n    res[\'Q3\'] = \'大きいほど T_0 が高い\' if s3 == \'＋\' else \'大きいほど T_0 が低い\' if s3 == \'−\' else \'はっきりしない\'\n    res[\'Q4\'] = \'学ぶほど T_0 が高い\' if s4 == \'＋\' else \'学ぶほど T_0 が低い\' if s4 == \'−\' else \'はっきりしない\'\n    # 記録：共通の ε\n    eps_c = float(np.median([b[2][\'eps\'] for b in blocks.values()]))\n    res[\'T0_common_eps\'] = {n: crossing_mono(np.median(b[1], 0), temps, eps_c) for n, b in blocks.items()}\n    res[\'eps_common\'] = eps_c\n    # 記録：A2-2 の 7 モデル（種だけのぶれ・古さとの切り分けの手がかり）\n    res[\'a22\'] = {}\n    for n, hf in A5_A22:\n        tok = AutoTokenizer.from_pretrained(hf)\n        pr = [split_doc(tok, d[\'text\']) for d in docs]\n        Z = (lambda z: {k: z[k] for k in z.files})(np.load(f\'{d2}/{n}.npz\'))\n        use = [int(u) for u in Z[\'use\']]\n        parts, fd, fs = model_block(Z, [pr[i][0] for i in use], [pr[i][1] for i in use], temps)\n        res[\'a22\'][n] = dict(T0=fs[\'T0\'], G07=G_at(parts, temps.index(0.7))[0], G_rel=G_rel_at(parts, temps, fs[\'T0\']),\n                             ent_T1_human=float(np.mean(Z[\'tf\'][2][:, temps.index(1.0), :])))\n    if \'pythia-1b\' in res[\'models\']:\n        a, b = res[\'models\'][\'pythia-1b\'], res[\'a22\'][\'pythia-1b\']\n        res[\'seed_spread_pythia1b\'] = {k: [a[k], b[k]] for k in [\'T0\', \'G07\', \'G_rel\']}\n    res[\'temps\'] = temps\n    return res\n\n\ndef self_test_a25_analyze():\n    assert abs(spearman([1, 2, 3, 4], [10, 20, 30, 40]) - 1) < 1e-9 and abs(spearman([1, 2, 3, 4], [4, 3, 2, 1]) + 1) < 1e-9\n    assert abs(spearman([1, 2, 3], [1, 1, 2]) - 0.8660254) < 1e-6          # 同じ値は平均の順位\n    # 軸の判定：点がきれいに並び、ブートストラップでも向きが同じなら「＋」\n    xs = np.log([70e6, 160e6, 410e6, 1e9, 1.4e9])\n    yb = np.array([[0.2, 0.3, 0.4, 0.5, 0.6]] * 100) + np.random.default_rng(0).normal(0, 0.01, (100, 5))\n    s, d = axis_word(xs, [0.2, 0.3, 0.4, 0.5, 0.6], yb, False, False, None, None)\n    assert s == \'＋\' and abs(d[\'rho\'] - 1) < 1e-9, (s, d)\n    s, d = axis_word(xs, [0.6, 0.5, 0.4, 0.3, 0.2], -yb, False, False, None, None)\n    assert s == \'−\'\n    s, d = axis_word(xs, [0.2, 0.3, np.nan, 0.5, 0.6], yb, False, False, None, None)\n    assert s == \'＋\' and d[\'n\'] == 4\n    s, d = axis_word(xs, [0.2, np.nan, np.nan, 0.5, 0.6], yb, False, False, None, None)\n    assert s == \'なし\'\n    noisy = np.random.default_rng(1).normal(0, 1, (100, 5))\n    s, d = axis_word(xs, [0.2, 0.3, 0.4, 0.5, 0.6], noisy, False, False, None, None)\n    assert s == \'なし\', d                                                  # 点はそろっても、向きが不確かなら言葉を付けない\n    st = [8000, 24000, 64000, 143000]\n    s, d = axis_word(st, [0.1, 0.2, 0.3, 0.4], np.array([[0.1, 0.2, 0.3, 0.4]] * 50), True, True, None, None)\n    assert s == \'＋\'\n    s, d = axis_word(st, [0.1, 0.3, 0.2, 0.4], np.array([[0.1, 0.2, 0.3, 0.4]] * 50), True, True, None, None)\n    assert s == \'なし\'                                                     # 学びの軸は完全にそろった並びだけ\n    s, d = axis_word(st, [0.1, np.nan, 0.3, 0.4], np.array([[0.1, 0.2, 0.3, 0.4]] * 50), True, True, None, None)\n    assert s == \'なし\'\n    xs5 = np.log([70e6, 160e6, 410e6, 1e9, 1.4e9])                       # ρ ＝ −0.9 ちょうど（1 つだけ入れ替わり）は「−」に入る\n    yb5 = np.array([[0.5, 0.4, 0.3, 0.1, 0.2]] * 50)\n    s, d = axis_word(xs5, [0.5, 0.4, 0.3, 0.1, 0.2], yb5, False, False, None, None)\n    assert s == \'−\' and abs(d[\'rho\'] + 0.9) < 1e-9, (s, d)\n    assert words(\'＋\', \'＋\', \'P\', \'N\') == \'P\' and words(\'＋\', \'なし\', \'P\', \'N\') == \'はっきりしない\'\n    # G と G_rel：作り物の parts で、補間と読まない条件\n    temps = [0.6, 0.7, 0.8, 0.85, 0.9, 0.95, 1.0, 1.05, 1.1, 1.15, 1.2, 1.25, 1.3, 1.4, 1.5]\n    parts = np.zeros((3, 40, 15))\n    parts[2] = -1.0; parts[0] = -0.4; parts[1] = -0.6                      # G ＝ 0.4 どこでも\n    assert abs(G_at(parts, 1)[0] - 0.4) < 1e-12 and abs(G_rel_at(parts, temps, 1.0) - 0.4) < 1e-12\n    p2 = parts.copy(); p2[2, :, 1] = -0.1; p2[0, :, 1] = -0.04             # f が −0.2 より上 → 読まない\n    assert np.isnan(G_at(p2, 1)[0])\n    p3 = parts.copy(); p3[:, 15:, 1] = np.nan                               # 使える記事 25 本 → 読まない\n    assert np.isnan(G_at(p3, 1)[0])\n    assert np.isnan(G_rel_at(parts, temps, float(\'nan\'))) and np.isnan(G_rel_at(parts, temps, 0.5))   # T* ＝ 0.4 は格子の外\n    return \'OK\'\n'
assert hashlib.sha256(SRC5A.encode()).hexdigest() == 'f4ef35b9f594a205ac1145c1a326e9de8a25280171fe2b93083cc1676e58d3ae'
exec(SRC22)
exec(SRCA22)
exec(SRCX)
exec(SRC5A)
print('自己テスト：', self_test_a25_analyze())


In [ ]:
# セル3：分析して保存（結果は表示しない）
import json
try:
    # A2-2 の分析のコード（SRCA22）が、同じ名前の SEED（ブートストラップの種 0）で記事を選ぶ種を上書きするので、
    # 記事を選ぶあいだだけ A2-2 の種（20261027）に戻す（2026-10-29 11:4x に直した。分析のコード a2_5_analyze.py は変えていない）
    _seed_keep = SEED
    SEED = 20261027
    docs, meta = load_docs()
    SEED = _seed_keep
    old = json.load(open('/content/drive/MyDrive/logit-thermo' + '/A2_2/docs_meta.json'))
    assert [d['text_sha256'] for d in old['docs']] == [d['text_sha256'] for d in meta['docs']], 'BBC の記事が A2-2 と違います'
    r = analyze5('/content/drive/MyDrive/logit-thermo/A2_5', '/content/drive/MyDrive/logit-thermo' + '/A2_2', docs)
    r['a2_5_analyze_sha256'] = 'f4ef35b9f594a205ac1145c1a326e9de8a25280171fe2b93083cc1676e58d3ae'
    json.dump(_clean(r), open('/content/drive/MyDrive/logit-thermo/A2_5' + '/a2_5_summary.json', 'w'), ensure_ascii=False, separators=(',', ':'))
    print('保存しました')
except Exception as e:
    import traceback
    _NOTIFIED['error'] = True
    txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
    v = os.environ.get('HF_TOKEN')
    if v: txt = txt.replace(v, '***')
    txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
    notify('⚠️ A2-5 の分析で止まりました。ランタイムを切ります：\n' + txt, tags='warning')
    if globals().get('AUTO_DISCONNECT', True):
        from google.colab import drive as _d, runtime as _r
        _d.flush_and_unmount(); time.sleep(3); _r.unassign()
    raise


In [ ]:
# 最後：終わりの通知と、ランタイムを切る
notify('🎉 終わりました：判定と要約を A2_5/a2_5_summary.json に保存しました', tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount(); time.sleep(3); _runtime.unassign()
